In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

In [2]:
import json
import pandas as pd

experiments_path = Path.cwd().parent / "experiments" / "model_comparison.json"
with open(experiments_path) as f:
    all_results = json.load(f)

# Exclude the dev-config RNN and the excluded-by-design BiLSTM from the
# PRODUCTION-CANDIDATE table, but keep them documented separately below.
production_candidates = {
    k: v for k, v in all_results.items()
    if "DEV CONFIG" not in k
}

table = pd.DataFrame({
    name: {metric: metrics.get(metric) for metric in ["MAE", "RMSE", "MAPE", "sMAPE", "R2"]}
    for name, metrics in production_candidates.items()
}).T

table["num_parameters"] = [
    all_results[name].get("num_parameters", "—") for name in table.index
]
table["training_time_seconds"] = [
    round(all_results[name].get("training_time_seconds", 0), 1) if all_results[name].get("training_time_seconds") else "—"
    for name in table.index
]

table = table.sort_values("MAE")
pd.set_option("display.width", 120)
print(table.to_string())

                            MAE        RMSE       MAPE      sMAPE        R2 num_parameters training_time_seconds
BiLSTM (experiment)   66.391543   91.860677   5.716332   5.598477  0.752458          47128                1109.4
LSTM                  67.422624   93.767627   5.773968   5.654699  0.742073          23576               15091.6
Stacked LSTM          69.655762   97.959247   5.974574   5.831754  0.718498          56856                1480.2
GRU                   71.905100   97.205096   6.156141   6.052415  0.722816          18072                 675.6
Transformer           73.772668  103.448335   6.356304   6.136836  0.686067          36376                1358.5
Simple RNN            76.722175  102.871061   6.573379   6.456432  0.689561           7064                 202.2
Univariate LSTM       79.758896  106.028036   6.864089   6.684010  0.670214          20248                 409.8
Attention-LSTM        86.769918  112.721887   7.356781   7.257780  0.627259          27800      

In [3]:
table.to_csv(Path.cwd().parent / "experiments" / "final_model_comparison.csv")
print("Saved experiments/final_model_comparison.csv")

Saved experiments/final_model_comparison.csv
